In [1]:
from pathlib import Path
import os

# Run the notebook from the root of your Formula-1 project.
PROJECT_ROOT = Path.cwd().resolve()

# Change this if your spark-warehouse folder is somewhere else.
WAREHOUSE_DIR = PROJECT_ROOT / "spark-warehouse"

print("Project root:", PROJECT_ROOT)
print("Spark warehouse:", WAREHOUSE_DIR)
print("Warehouse exists:", WAREHOUSE_DIR.exists())

Project root: /sfs/gpfs/tardis/home/azd9vv/Formula-1
Spark warehouse: /sfs/gpfs/tardis/home/azd9vv/Formula-1/spark-warehouse
Warehouse exists: True


In [10]:
from pathlib import Path

DATABASE_NAME = "formula1"

PROJECT_ROOT = Path.cwd().resolve()
WAREHOUSE_DIRECTORY = PROJECT_ROOT / "spark-warehouse"
DATABASE_DIRECTORY = WAREHOUSE_DIRECTORY / "formula1.db"

print("Current directory:", PROJECT_ROOT)
print("Warehouse:", WAREHOUSE_DIRECTORY)
print("Database:", DATABASE_DIRECTORY)
print("Database exists:", DATABASE_DIRECTORY.exists())

Current directory: /sfs/gpfs/tardis/home/azd9vv/Formula-1
Warehouse: /sfs/gpfs/tardis/home/azd9vv/Formula-1/spark-warehouse
Database: /sfs/gpfs/tardis/home/azd9vv/Formula-1/spark-warehouse/formula1.db
Database exists: True


In [12]:
if not DATABASE_DIRECTORY.exists():
    raise FileNotFoundError(
        f"Database directory not found: {DATABASE_DIRECTORY}"
    )

table_directories = sorted(
    path
    for path in DATABASE_DIRECTORY.iterdir()
    if path.is_dir()
)

for table_directory in table_directories:
    print(table_directory.name)

drivers
laps
results
session_info
session_status
track_status
weather_data


In [13]:
spark.sql(
    f"""
    CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
    LOCATION '{DATABASE_DIRECTORY.as_uri()}'
    """
)

spark.sql(f"USE {DATABASE_NAME}")

spark.sql("SELECT current_database()").show()

26/07/10 23:56:41 WARN ObjectStore: Failed to get database formula1, returning NoSuchObjectException
26/07/10 23:56:41 WARN ObjectStore: Failed to get database formula1, returning NoSuchObjectException
26/07/10 23:56:41 WARN ObjectStore: Failed to get database formula1, returning NoSuchObjectException


+----------------+
|current_schema()|
+----------------+
|        formula1|
+----------------+



In [14]:
for table_directory in table_directories:
    table_name = table_directory.name

    if table_name.startswith((".", "_")):
        continue

    storage_format = (
        "DELTA"
        if (table_directory / "_delta_log").exists()
        else "PARQUET"
    )

    try:
        spark.sql(
            f"""
            CREATE TABLE IF NOT EXISTS
                {DATABASE_NAME}.`{table_name}`
            USING {storage_format}
            LOCATION '{table_directory.as_uri()}'
            """
        )

        print(
            f"Registered {DATABASE_NAME}.{table_name} "
            f"using {storage_format}"
        )

    except Exception as error:
        print(f"Could not register {table_name}: {error}")

26/07/10 23:56:48 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/07/10 23:56:49 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


Registered formula1.drivers using PARQUET
Registered formula1.laps using PARQUET
Registered formula1.results using PARQUET
Registered formula1.session_info using PARQUET
Registered formula1.session_status using PARQUET
Registered formula1.track_status using PARQUET
Registered formula1.weather_data using PARQUET


In [15]:
spark.sql("SHOW TABLES IN formula1").show(truncate=False)

+---------+--------------+-----------+
|namespace|tableName     |isTemporary|
+---------+--------------+-----------+
|formula1 |drivers       |false      |
|formula1 |laps          |false      |
|formula1 |results       |false      |
|formula1 |session_info  |false      |
|formula1 |session_status|false      |
|formula1 |track_status  |false      |
|formula1 |weather_data  |false      |
+---------+--------------+-----------+



In [16]:
DATABASE_NAME = "formula1"

if spark.catalog.databaseExists(DATABASE_NAME):
    spark.sql(f"USE {DATABASE_NAME}")
    print(f"Connected to database: {DATABASE_NAME}")
else:
    print(f"Database '{DATABASE_NAME}' was not found.")

Connected to database: formula1


In [17]:
DATABASE_NAME = "formula1"
DATABASE_PATH = WAREHOUSE_DIR / f"{DATABASE_NAME}.db"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")
spark.sql(f"USE {DATABASE_NAME}")

required_tables = [
    "results",
    "laps",
    "weather_data",
    "drivers",
    "session_info",
    "session_status",
    "track_status"
]

for table_name in required_tables:
    table_path = DATABASE_PATH / table_name

    if table_path.exists():
        spark.sql(
            f"""
            CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.{table_name}
            USING PARQUET
            LOCATION '{table_path.as_uri()}'
            """
        )

        print(f"Connected: {DATABASE_NAME}.{table_name}")
    else:
        print(f"Not found: {table_path}")

Connected: formula1.results
Connected: formula1.laps
Connected: formula1.weather_data
Connected: formula1.drivers
Connected: formula1.session_info
Connected: formula1.session_status
Connected: formula1.track_status


In [18]:
required_model_tables = [
    "results",
    "laps",
    "weather_data"
]

missing_tables = [
    table_name
    for table_name in required_model_tables
    if not spark.catalog.tableExists(
        f"formula1.{table_name}"
    )
]

if missing_tables:
    raise RuntimeError(
        "Missing required tables: "
        + ", ".join(missing_tables)
    )

print("All required tables are available.")

All required tables are available.


In [19]:
results = spark.table("formula1.results")
laps = spark.table("formula1.laps")
weather_data = spark.table("formula1.weather_data")

print("Results rows:", results.count())
print("Laps rows:", laps.count())
print("Weather rows:", weather_data.count())

Results rows: 5526
Laps rows: 193572
Weather rows: 33398


In [20]:
print("--- RESULTS ---")
results.printSchema()

print("--- LAPS ---")
laps.printSchema()

print("--- WEATHER DATA ---")
weather_data.printSchema()

--- RESULTS ---
root
 |-- DriverNumber: integer (nullable = true)
 |-- BroadcastName: string (nullable = true)
 |-- Abbreviation: string (nullable = true)
 |-- DriverId: string (nullable = true)
 |-- TeamName: string (nullable = true)
 |-- TeamColor: string (nullable = true)
 |-- TeamId: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- FullName: string (nullable = true)
 |-- HeadshotUrl: string (nullable = true)
 |-- CountryCode: string (nullable = true)
 |-- Position: double (nullable = true)
 |-- ClassifiedPosition: string (nullable = true)
 |-- GridPosition: double (nullable = true)
 |-- Q1: string (nullable = true)
 |-- Q2: string (nullable = true)
 |-- Q3: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Points: double (nullable = true)
 |-- Laps: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- session_nu

In [21]:
DATABASE_NAME = "formula1"

spark.sql(f"USE {DATABASE_NAME}")

results = spark.table(f"{DATABASE_NAME}.results")
laps = spark.table(f"{DATABASE_NAME}.laps")
weather = spark.table(f"{DATABASE_NAME}.weather_data")

print("Results:", results.count())
print("Laps:", laps.count())
print("Weather:", weather.count())

Results: 5526
Laps: 193572
Weather: 33398


In [22]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW results_clean AS

SELECT
    CAST(year AS INT) AS year,
    CAST(session_number AS INT) AS session_number,
    UPPER(TRIM(CAST(type_of_race AS STRING))) AS session_type,

    CAST(DriverNumber AS STRING) AS driver_number,

    COALESCE(
        NULLIF(TRIM(CAST(DriverId AS STRING)), ''),
        NULLIF(TRIM(CAST(Abbreviation AS STRING)), ''),
        CAST(DriverNumber AS STRING)
    ) AS driver_id,

    NULLIF(TRIM(CAST(TeamName AS STRING)), '') AS team_name,

    TRY_CAST(Position AS INT) AS result_position,
    TRY_CAST(GridPosition AS INT) AS grid_position,
    TRY_CAST(Points AS DOUBLE) AS championship_points,

    CAST(Status AS STRING) AS status

FROM formula1.results
""")

26/07/11 00:02:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[]

In [23]:
spark.sql("""
SELECT
    year,
    session_type,
    COUNT(*) AS rows,
    COUNT(DISTINCT session_number) AS sessions
FROM results_clean
GROUP BY year, session_type
ORDER BY year, session_type
""").show(100)

[Stage 26:>                                                       (0 + 23) / 23]

+----+------------+----+--------+
|year|session_type|rows|sessions|
+----+------------+----+--------+
|2018|           Q| 420|      21|
|2019|           Q| 420|      21|
|2020|           Q| 340|      17|
|2020|           R| 340|      17|
|2021|           Q| 440|      22|
|2021|           R| 440|      22|
|2022|           Q| 440|      22|
|2022|           R| 440|      22|
|2023|           Q|  60|       3|
|2023|           R| 440|      22|
|2024|           R| 479|      24|
|2025|           Q| 480|      24|
|2025|           R| 479|      24|
|2026|           Q| 154|       7|
|2026|           R| 154|       7|
+----+------------+----+--------+



In [25]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW laps_clean AS

SELECT
    CAST(year AS INT) AS year,
    CAST(session_number AS INT) AS session_number,
    UPPER(TRIM(CAST(type_of_race AS STRING))) AS session_type,

    CAST(DriverNumber AS STRING) AS driver_number,
    CAST(Driver AS STRING) AS driver_abbreviation,
    CAST(Team AS STRING) AS team_name,

    TRY_CAST(LapNumber AS INT) AS lap_number,
    TRY_CAST(Position AS DOUBLE) AS running_position,
    TRY_CAST(Stint AS INT) AS stint,

    CAST(Compound AS STRING) AS compound,

    TRY_CAST(SpeedI1 AS DOUBLE) AS speed_i1,
    TRY_CAST(SpeedI2 AS DOUBLE) AS speed_i2,
    TRY_CAST(SpeedFL AS DOUBLE) AS finish_line_speed,
    TRY_CAST(SpeedST AS DOUBLE) AS speed_trap,

    CASE
        WHEN LapTime IS NULL
          OR LOWER(TRIM(CAST(LapTime AS STRING)))
             IN ('', 'nat', 'null', 'none')
        THEN NULL

        ELSE
            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(LapTime AS STRING),
                            '([0-9]+) days',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 86400

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(LapTime AS STRING),
                            'days ([0-9]{2}):',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 3600

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(LapTime AS STRING),
                            'days [0-9]{2}:([0-9]{2}):',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 60

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(LapTime AS STRING),
                            'days [0-9]{2}:[0-9]{2}:([0-9]+[.]?[0-9]*)',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            )
    END AS lap_time_seconds

FROM formula1.laps
""")

DataFrame[]

In [26]:
spark.sql("""
SELECT
    year,
    session_number,
    session_type,
    driver_number,
    lap_number,
    lap_time_seconds
FROM laps_clean
WHERE lap_time_seconds IS NOT NULL
ORDER BY year, session_number, driver_number, lap_number
LIMIT 30
""").show()

[Stage 32:==============================>                         (11 + 9) / 20]

+----+--------------+------------+-------------+----------+------------------+
|year|session_number|session_type|driver_number|lap_number|  lap_time_seconds|
+----+--------------+------------+-------------+----------+------------------+
|2018|             1|           Q|           10|         1|             120.4|
|2018|             1|           Q|           10|         2|            85.295|
|2018|             1|           Q|           10|         3|           108.298|
|2018|             1|           Q|           10|         4|            85.425|
|2018|             1|           Q|           10|         6|           117.672|
|2018|             1|           Q|           11|         1|109.61500000000001|
|2018|             1|           Q|           11|         2|            84.914|
|2018|             1|           Q|           11|         3|           111.866|
|2018|             1|           Q|           11|         4|            85.005|
|2018|             1|           Q|           11|    

In [27]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW race_outcomes AS

SELECT
    year,
    session_number,
    driver_number,
    driver_id,
    team_name,

    result_position AS target_position,
    championship_points AS target_championship_points,

    grid_position,
    status,

    CASE
        WHEN grid_position > 0
         AND result_position IS NOT NULL
        THEN grid_position - result_position
        ELSE NULL
    END AS positions_gained,

    CASE
        WHEN status IS NULL THEN NULL
        WHEN LOWER(status) = 'finished' THEN 0
        WHEN LOWER(status) LIKE '+%lap%' THEN 0
        ELSE 1
    END AS dnf_flag

FROM results_clean

WHERE session_type = 'R'
  AND result_position IS NOT NULL
""")

DataFrame[]

In [28]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW qualifying_results AS

WITH qualifying AS (
    SELECT
        year,
        session_number,
        driver_number,
        driver_id,
        team_name,
        result_position AS qualifying_position
    FROM results_clean
    WHERE session_type = 'Q'
      AND result_position IS NOT NULL
),

team_windows AS (
    SELECT
        *,

        COUNT(*) OVER (
            PARTITION BY year, session_number
        ) AS qualifying_field_size,

        COUNT(*) OVER (
            PARTITION BY year, session_number, team_name
        ) AS team_driver_count,

        SUM(qualifying_position) OVER (
            PARTITION BY year, session_number, team_name
        ) AS team_position_sum

    FROM qualifying
)

SELECT
    year,
    session_number,
    driver_number,
    driver_id,
    team_name,

    qualifying_position,
    qualifying_field_size,

    CASE
        WHEN qualifying_field_size > 1
        THEN
            (qualifying_position - 1.0)
            / (qualifying_field_size - 1.0)
    END AS qualifying_position_percentile,

    CASE
        WHEN qualifying_position <= 10 THEN 1
        ELSE 0
    END AS qualified_top_10,

    CASE
        WHEN team_driver_count = 2
        THEN team_position_sum - qualifying_position
    END AS teammate_qualifying_position,

    CASE
        WHEN team_driver_count = 2
        THEN
            qualifying_position
            - (team_position_sum - qualifying_position)
    END AS qualifying_vs_teammate

FROM team_windows
""")

DataFrame[]

In [29]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW qualifying_lap_features AS

SELECT
    year,
    session_number,
    driver_number,

    COUNT(lap_time_seconds) AS valid_qualifying_laps,

    MIN(lap_time_seconds) AS best_qualifying_lap,

    PERCENTILE_APPROX(
        lap_time_seconds,
        0.50
    ) AS median_qualifying_lap,

    AVG(lap_time_seconds) AS mean_qualifying_lap,

    STDDEV_SAMP(
        lap_time_seconds
    ) AS qualifying_lap_sd,

    MAX(speed_i1) AS maximum_speed_i1,
    MAX(speed_i2) AS maximum_speed_i2,
    MAX(finish_line_speed) AS maximum_finish_line_speed,
    MAX(speed_trap) AS maximum_speed_trap,

    COUNT(DISTINCT stint) AS qualifying_stints,
    COUNT(DISTINCT compound) AS qualifying_compounds

FROM laps_clean

WHERE session_type = 'Q'
  AND lap_time_seconds BETWEEN 20 AND 400

GROUP BY
    year,
    session_number,
    driver_number
""")

DataFrame[]

In [30]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW qualifying_features AS

WITH combined AS (
    SELECT
        q.*,

        l.valid_qualifying_laps,
        l.best_qualifying_lap,
        l.median_qualifying_lap,
        l.mean_qualifying_lap,
        l.qualifying_lap_sd,

        l.maximum_speed_i1,
        l.maximum_speed_i2,
        l.maximum_finish_line_speed,
        l.maximum_speed_trap,

        l.qualifying_stints,
        l.qualifying_compounds

    FROM qualifying_results q

    LEFT JOIN qualifying_lap_features l
      ON q.year = l.year
     AND q.session_number = l.session_number
     AND q.driver_number = l.driver_number
)

SELECT
    *,

    best_qualifying_lap
    - MIN(best_qualifying_lap) OVER (
        PARTITION BY year, session_number
    ) AS qualifying_gap_to_pole

FROM combined
""")

DataFrame[]

In [31]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW qualifying_weather AS

SELECT
    CAST(year AS INT) AS year,
    CAST(session_number AS INT) AS session_number,

    AVG(TRY_CAST(AirTemp AS DOUBLE)) AS qualifying_air_temperature,
    AVG(TRY_CAST(TrackTemp AS DOUBLE)) AS qualifying_track_temperature,
    AVG(TRY_CAST(Humidity AS DOUBLE)) AS qualifying_humidity,
    AVG(TRY_CAST(Pressure AS DOUBLE)) AS qualifying_pressure,
    AVG(TRY_CAST(WindSpeed AS DOUBLE)) AS qualifying_wind_speed,

    MAX(
        CASE
            WHEN LOWER(TRIM(CAST(Rainfall AS STRING)))
                 IN ('true', '1', '1.0')
            THEN 1
            ELSE 0
        END
    ) AS qualifying_rain

FROM formula1.weather_data

WHERE UPPER(TRIM(CAST(type_of_race AS STRING))) = 'Q'

GROUP BY
    CAST(year AS INT),
    CAST(session_number AS INT)
""")

DataFrame[]

In [32]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW driver_history AS

SELECT
    r.*,

    LAG(target_position, 1) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
    ) AS previous_finish,

    LAG(target_championship_points, 1) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
    ) AS previous_points,

    LAG(dnf_flag, 1) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
    ) AS previous_dnf,

    AVG(target_position) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
        ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
    ) AS average_finish_last_3,

    AVG(target_position) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS average_finish_last_5,

    STDDEV_SAMP(target_position) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS finish_sd_last_5,

    AVG(COALESCE(target_championship_points, 0.0)) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS average_points_last_5,

    SUM(COALESCE(target_championship_points, 0.0)) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS total_points_last_5,

    AVG(dnf_flag) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS dnf_rate_last_5,

    AVG(positions_gained) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS average_positions_gained_last_5,

    SUM(COALESCE(target_championship_points, 0.0)) OVER (
        PARTITION BY driver_id, year
        ORDER BY session_number
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS season_points_before_race,

    COUNT(*) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS career_races_before

FROM race_outcomes r
""")

DataFrame[]

In [33]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW qualifying_history AS

SELECT
    q.*,

    AVG(qualifying_position) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
        ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
    ) AS average_qualifying_position_last_3,

    AVG(qualifying_position) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS average_qualifying_position_last_5,

    AVG(qualifying_gap_to_pole) OVER (
        PARTITION BY driver_id
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS average_gap_to_pole_last_5

FROM qualifying_features q
""")

DataFrame[]

In [34]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW team_race_results AS

SELECT
    year,
    session_number,
    team_name,

    AVG(target_position) AS team_event_average_finish,

    SUM(
        COALESCE(target_championship_points, 0.0)
    ) AS team_event_points,

    AVG(dnf_flag) AS team_event_dnf_rate

FROM race_outcomes

WHERE team_name IS NOT NULL

GROUP BY
    year,
    session_number,
    team_name
""")

DataFrame[]

In [35]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW team_history AS

SELECT
    t.*,

    AVG(team_event_average_finish) OVER (
        PARTITION BY team_name
        ORDER BY year, session_number
        ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
    ) AS team_average_finish_last_3,

    AVG(team_event_average_finish) OVER (
        PARTITION BY team_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS team_average_finish_last_5,

    AVG(team_event_points) OVER (
        PARTITION BY team_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS team_average_points_last_5,

    AVG(team_event_dnf_rate) OVER (
        PARTITION BY team_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS team_dnf_rate_last_5

FROM team_race_results t
""")

DataFrame[]

In [36]:
modeling_df = spark.sql("""
SELECT
    d.year,
    d.session_number,

    d.driver_number,
    d.driver_id,
    d.team_name,

    q.qualifying_position,
    q.qualifying_field_size,
    q.qualifying_position_percentile,
    q.qualified_top_10,
    q.teammate_qualifying_position,
    q.qualifying_vs_teammate,

    q.valid_qualifying_laps,
    q.best_qualifying_lap,
    q.median_qualifying_lap,
    q.mean_qualifying_lap,
    q.qualifying_lap_sd,
    q.qualifying_gap_to_pole,

    q.maximum_speed_i1,
    q.maximum_speed_i2,
    q.maximum_finish_line_speed,
    q.maximum_speed_trap,

    q.qualifying_stints,
    q.qualifying_compounds,

    q.average_qualifying_position_last_3,
    q.average_qualifying_position_last_5,
    q.average_gap_to_pole_last_5,

    w.qualifying_air_temperature,
    w.qualifying_track_temperature,
    w.qualifying_humidity,
    w.qualifying_pressure,
    w.qualifying_wind_speed,
    w.qualifying_rain,

    d.previous_finish,
    d.previous_points,
    d.previous_dnf,

    d.average_finish_last_3,
    d.average_finish_last_5,
    d.finish_sd_last_5,

    d.average_points_last_5,
    d.total_points_last_5,
    d.dnf_rate_last_5,
    d.average_positions_gained_last_5,

    d.season_points_before_race,
    d.career_races_before,

    t.team_average_finish_last_3,
    t.team_average_finish_last_5,
    t.team_average_points_last_5,
    t.team_dnf_rate_last_5,

    d.target_position,
    d.target_championship_points

FROM driver_history d

INNER JOIN qualifying_history q
  ON d.year = q.year
 AND d.session_number = q.session_number
 AND d.driver_number = q.driver_number

LEFT JOIN qualifying_weather w
  ON d.year = w.year
 AND d.session_number = w.session_number

LEFT JOIN team_history t
  ON d.year = t.year
 AND d.session_number = t.session_number
 AND d.team_name = t.team_name
""")

In [37]:
print("Rows:", modeling_df.count())
print("Columns:", len(modeling_df.columns))

modeling_df.printSchema()

Rows: 1906
Columns: 50
root
 |-- year: integer (nullable = true)
 |-- session_number: integer (nullable = true)
 |-- driver_number: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- team_name: string (nullable = true)
 |-- qualifying_position: integer (nullable = true)
 |-- qualifying_field_size: long (nullable = false)
 |-- qualifying_position_percentile: decimal(36,24) (nullable = true)
 |-- qualified_top_10: integer (nullable = false)
 |-- teammate_qualifying_position: long (nullable = true)
 |-- qualifying_vs_teammate: long (nullable = true)
 |-- valid_qualifying_laps: long (nullable = true)
 |-- best_qualifying_lap: double (nullable = true)
 |-- median_qualifying_lap: double (nullable = true)
 |-- mean_qualifying_lap: double (nullable = true)
 |-- qualifying_lap_sd: double (nullable = true)
 |-- qualifying_gap_to_pole: double (nullable = true)
 |-- maximum_speed_i1: double (nullable = true)
 |-- maximum_speed_i2: double (nullable = true)
 |-- maximum_finish_li

In [38]:
modeling_df.orderBy(
    "year",
    "session_number",
    "target_position"
).show(30, truncate=False)

+----+--------------+-------------+---------------+-----------------+-------------------+---------------------+------------------------------+----------------+----------------------------+----------------------+---------------------+-------------------+---------------------+-------------------+------------------+----------------------+----------------+----------------+-------------------------+------------------+-----------------+--------------------+----------------------------------+----------------------------------+--------------------------+--------------------------+----------------------------+-------------------+-------------------+---------------------+---------------+---------------+---------------+------------+---------------------+---------------------+----------------+---------------------+-------------------+---------------+-------------------------------+-------------------------+-------------------+--------------------------+--------------------------+------------------

In [39]:
duplicate_rows = (
    modeling_df
    .groupBy(
        "year",
        "session_number",
        "driver_number"
    )
    .count()
    .filter("count > 1")
)

duplicate_rows.show(truncate=False)

+----+--------------+-------------+-----+
|year|session_number|driver_number|count|
+----+--------------+-------------+-----+
+----+--------------+-------------+-----+



In [40]:
from pyspark.sql import functions as F

missing_counts = modeling_df.select(
    *[
        F.sum(
            F.col(column).isNull().cast("int")
        ).alias(column)
        for column in modeling_df.columns
    ]
)

missing_counts.show(truncate=False, vertical=True)

-RECORD 0---------------------------------
 year                               | 0   
 session_number                     | 0   
 driver_number                      | 0   
 driver_id                          | 0   
 team_name                          | 0   
 qualifying_position                | 0   
 qualifying_field_size              | 0   
 qualifying_position_percentile     | 0   
 qualified_top_10                   | 0   
 teammate_qualifying_position       | 5   
 qualifying_vs_teammate             | 5   
 valid_qualifying_laps              | 11  
 best_qualifying_lap                | 11  
 median_qualifying_lap              | 11  
 mean_qualifying_lap                | 11  
 qualifying_lap_sd                  | 21  
 qualifying_gap_to_pole             | 11  
 maximum_speed_i1                   | 11  
 maximum_speed_i2                   | 11  
 maximum_finish_line_speed          | 13  
 maximum_speed_trap                 | 11  
 qualifying_stints                  | 11  
 qualifying

In [74]:
modeling_df.groupBy("year").agg(
    F.count("*").alias("rows"),
    F.countDistinct("session_number").alias("races"),
    F.countDistinct("driver_id").alias("drivers")
).orderBy("year").show()

+----+----+-----+-------+
|year|rows|races|drivers|
+----+----+-----+-------+
|2020| 340|   17|     23|
|2021| 438|   22|     21|
|2022| 439|   22|     22|
|2023|  60|    3|     20|
|2025| 478|   24|     21|
|2026| 151|    7|     22|
+----+----+-----+-------+



In [75]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW race_weather AS

SELECT
    CAST(year AS INT) AS year,
    CAST(session_number AS INT) AS session_number,

    AVG(
        TRY_CAST(AirTemp AS DOUBLE)
    ) AS race_air_temperature,

    AVG(
        TRY_CAST(TrackTemp AS DOUBLE)
    ) AS race_track_temperature,

    AVG(
        TRY_CAST(Humidity AS DOUBLE)
    ) AS race_humidity,

    AVG(
        TRY_CAST(Pressure AS DOUBLE)
    ) AS race_pressure,

    AVG(
        TRY_CAST(WindSpeed AS DOUBLE)
    ) AS race_wind_speed,

    MAX(
        TRY_CAST(WindSpeed AS DOUBLE)
    ) AS race_max_wind_speed,

    MIN(
        TRY_CAST(AirTemp AS DOUBLE)
    ) AS race_min_air_temperature,

    MAX(
        TRY_CAST(AirTemp AS DOUBLE)
    ) AS race_max_air_temperature,

    MAX(
        CASE
            WHEN LOWER(TRIM(CAST(Rainfall AS STRING)))
                 IN ('true', '1', '1.0')
            THEN 1
            ELSE 0
        END
    ) AS race_rain,

    AVG(
        CASE
            WHEN LOWER(TRIM(CAST(Rainfall AS STRING)))
                 IN ('true', '1', '1.0')
            THEN 1.0
            ELSE 0.0
        END
    ) AS race_rain_fraction

FROM formula1.weather_data

WHERE UPPER(TRIM(CAST(type_of_race AS STRING))) = 'R'

GROUP BY
    CAST(year AS INT),
    CAST(session_number AS INT)
""")

DataFrame[]

In [76]:
spark.sql("""
SELECT
    year,
    COUNT(*) AS race_weather_sessions,
    SUM(race_rain) AS rainy_sessions
FROM race_weather
GROUP BY year
ORDER BY year
""").show()

+----+---------------------+--------------+
|year|race_weather_sessions|rainy_sessions|
+----+---------------------+--------------+
|2020|                   17|             4|
|2021|                   22|             6|
|2022|                   22|             6|
|2023|                   22|             5|
|2024|                   24|             5|
|2025|                   24|             6|
|2026|                    7|             1|
+----+---------------------+--------------+



In [42]:
spark.sql("""
DROP TABLE IF EXISTS formula1.driver_race_modeling
""")

DataFrame[]

In [77]:
modeling_df = spark.sql("""
SELECT
    d.year,
    d.session_number,

    d.driver_number,
    d.driver_id,
    d.team_name,

    q.qualifying_position,
    q.qualifying_field_size,
    q.qualifying_position_percentile,
    q.qualified_top_10,
    q.teammate_qualifying_position,
    q.qualifying_vs_teammate,

    q.valid_qualifying_laps,
    q.best_qualifying_lap,
    q.median_qualifying_lap,
    q.mean_qualifying_lap,
    q.qualifying_lap_sd,
    q.qualifying_gap_to_pole,

    q.maximum_speed_i1,
    q.maximum_speed_i2,
    q.maximum_finish_line_speed,
    q.maximum_speed_trap,

    q.qualifying_stints,
    q.qualifying_compounds,

    q.average_qualifying_position_last_3,
    q.average_qualifying_position_last_5,
    q.average_gap_to_pole_last_5,

    w.race_air_temperature,
    w.race_track_temperature,
    w.race_humidity,
    w.race_pressure,
    w.race_wind_speed,
    w.race_max_wind_speed,
    w.race_min_air_temperature,
    w.race_max_air_temperature,
    w.race_rain,
    w.race_rain_fraction,

    d.previous_finish,
    d.previous_points,
    d.previous_dnf,

    d.average_finish_last_3,
    d.average_finish_last_5,
    d.finish_sd_last_5,

    d.average_points_last_5,
    d.total_points_last_5,
    d.dnf_rate_last_5,
    d.average_positions_gained_last_5,

    d.season_points_before_race,
    d.career_races_before,

    t.team_average_finish_last_3,
    t.team_average_finish_last_5,
    t.team_average_points_last_5,
    t.team_dnf_rate_last_5,

    d.target_position,
    d.target_championship_points

FROM driver_history d

INNER JOIN qualifying_history q
  ON d.year = q.year
 AND d.session_number = q.session_number
 AND d.driver_number = q.driver_number

LEFT JOIN race_weather w
  ON d.year = w.year
 AND d.session_number = w.session_number

LEFT JOIN team_history t
  ON d.year = t.year
 AND d.session_number = t.session_number
 AND d.team_name = t.team_name
""")

In [78]:
(
    modeling_df
    .write
    .mode("overwrite")
    .format("parquet")
    .saveAsTable("formula1.driver_race_modeling")
)

In [44]:
spark.sql("""
SELECT
    year,
    COUNT(*) AS rows,
    COUNT(DISTINCT session_number) AS races
FROM formula1.driver_race_modeling
GROUP BY year
ORDER BY year
""").show()

+----+----+-----+
|year|rows|races|
+----+----+-----+
|2020| 340|   17|
|2021| 438|   22|
|2022| 439|   22|
|2023|  60|    3|
|2025| 478|   24|
|2026| 151|    7|
+----+----+-----+



In [45]:
model_data = spark.table(
    "formula1.driver_race_modeling"
)

model_data.show(5, truncate=False)

+----+--------------+-------------+---------+---------------+-------------------+---------------------+------------------------------+----------------+----------------------------+----------------------+---------------------+-------------------+---------------------+-------------------+------------------+----------------------+----------------+----------------+-------------------------+------------------+-----------------+--------------------+----------------------------------+----------------------------------+--------------------------+--------------------------+----------------------------+-------------------+-------------------+---------------------+---------------+---------------+---------------+------------+---------------------+---------------------+-----------------+---------------------+-------------------+------------------+-------------------------------+-------------------------+-------------------+--------------------------+--------------------------+----------------------

In [46]:
train_data = model_data.filter(
    F.col("year") <= 2023
)

validation_data = model_data.filter(
    F.col("year") == 2024
)

test_data = model_data.filter(
    F.col("year") >= 2025
)

print("Training rows:", train_data.count())
print("Validation rows:", validation_data.count())
print("Test rows:", test_data.count())

Training rows: 1277
Validation rows: 0
Test rows: 629


In [58]:
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Imputer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [59]:
model_data = (
    spark.table("formula1.driver_race_modeling")
    .filter(F.col("target_position").isNotNull())
)

LABEL = "target_position"

print("Rows:", model_data.count())

Rows: 1906


In [60]:
available_years = [
    row["year"]
    for row in (
        model_data
        .select("year")
        .distinct()
        .orderBy("year")
        .collect()
    )
]

if len(available_years) < 3:
    raise ValueError(
        "At least three seasons are needed for "
        "training, validation, and testing."
    )

VALIDATION_YEAR = available_years[-2]
TEST_YEAR = available_years[-1]

train_data = (
    model_data
    .filter(F.col("year") < VALIDATION_YEAR)
    .cache()
)

validation_data = (
    model_data
    .filter(F.col("year") == VALIDATION_YEAR)
    .cache()
)

test_data = (
    model_data
    .filter(F.col("year") == TEST_YEAR)
    .cache()
)

print("Available years:", available_years)
print("Validation year:", VALIDATION_YEAR)
print("Test year:", TEST_YEAR)

print("Training rows:", train_data.count())
print("Validation rows:", validation_data.count())
print("Test rows:", test_data.count())

Available years: [2020, 2021, 2022, 2023, 2025, 2026]
Validation year: 2025
Test year: 2026
Training rows: 1277
Validation rows: 478
Test rows: 151


In [47]:
LABEL = "target_position"

In [48]:
LABEL = "target_championship_points"

In [79]:
requested_numeric_features = [
    "session_number",

    # Current qualifying result
    "qualifying_position",
    "qualifying_field_size",
    "qualifying_position_percentile",
    "qualified_top_10",
    "teammate_qualifying_position",
    "qualifying_vs_teammate",

    # Current qualifying performance
    "valid_qualifying_laps",
    "best_qualifying_lap",
    "median_qualifying_lap",
    "mean_qualifying_lap",
    "qualifying_lap_sd",
    "qualifying_gap_to_pole",

    # Qualifying speed
    "maximum_speed_i1",
    "maximum_speed_i2",
    "maximum_finish_line_speed",
    "maximum_speed_trap",

    # Qualifying session usage
    "qualifying_stints",
    "qualifying_compounds",

    # Historical qualifying form
    "average_qualifying_position_last_3",
    "average_qualifying_position_last_5",
    "average_gap_to_pole_last_5",

    # Actual race weather
    "race_air_temperature",
    "race_track_temperature",
    "race_humidity",
    "race_pressure",
    "race_wind_speed",
    "race_max_wind_speed",
    "race_min_air_temperature",
    "race_max_air_temperature",
    "race_rain",
    "race_rain_fraction",

    # Previous race
    "previous_finish",
    "previous_points",
    "previous_dnf",

    # Historical driver form
    "average_finish_last_3",
    "average_finish_last_5",
    "finish_sd_last_5",
    "average_points_last_5",
    "total_points_last_5",
    "dnf_rate_last_5",
    "average_positions_gained_last_5",

    # Season and career
    "season_points_before_race",
    "career_races_before",

    # Historical team performance
    "team_average_finish_last_3",
    "team_average_finish_last_5",
    "team_average_points_last_5",
    "team_dnf_rate_last_5"
]

categorical_features = [
    "driver_id",
    "team_name"
]

In [80]:
numeric_features = [
    column
    for column in requested_numeric_features
    if column in model_data.columns
]

categorical_features = [
    column
    for column in categorical_features
    if column in model_data.columns
]

missing_features = sorted(
    set(requested_numeric_features)
    - set(numeric_features)
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)
print("Missing optional features:", missing_features)

Numeric features: 38
Categorical features: ['driver_id', 'team_name']
Missing optional features: ['race_air_temperature', 'race_humidity', 'race_max_air_temperature', 'race_max_wind_speed', 'race_min_air_temperature', 'race_pressure', 'race_rain', 'race_rain_fraction', 'race_track_temperature', 'race_wind_speed']


In [81]:
for column in numeric_features:
    model_data = model_data.withColumn(
        column,
        F.col(column).cast("double")
    )

In [82]:
def create_regression_pipeline(
    label_col,
    reg_param,
    elastic_net_param
):
    imputed_columns = [
        f"{column}_imputed"
        for column in numeric_features
    ]

    imputer = Imputer(
        inputCols=numeric_features,
        outputCols=imputed_columns,
        strategy="median"
    )

    indexers = [
        StringIndexer(
            inputCol=column,
            outputCol=f"{column}_index",
            handleInvalid="keep"
        )
        for column in categorical_features
    ]

    encoder = OneHotEncoder(
        inputCols=[
            f"{column}_index"
            for column in categorical_features
        ],
        outputCols=[
            f"{column}_ohe"
            for column in categorical_features
        ],
        handleInvalid="keep",
        dropLast=True
    )

    assembler_inputs = (
        imputed_columns
        + [
            f"{column}_ohe"
            for column in categorical_features
        ]
    )

    assembler = VectorAssembler(
        inputCols=assembler_inputs,
        outputCol="features",
        handleInvalid="keep"
    )

    regression = LinearRegression(
        featuresCol="features",
        labelCol=label_col,
        predictionCol="prediction",

        regParam=reg_param,
        elasticNetParam=elastic_net_param,

        fitIntercept=True,
        standardization=True,
        solver="auto",
        maxIter=1000,
        tol=1e-6
    )

    return Pipeline(
        stages=[
            imputer,
            *indexers,
            encoder,
            assembler,
            regression
        ]
    )

In [83]:
def regression_metrics(
    fitted_pipeline,
    data,
    label_col
):
    predictions = fitted_pipeline.transform(data)

    metric_values = {}

    for metric_name in ["rmse", "mae", "r2"]:
        evaluator = RegressionEvaluator(
            labelCol=label_col,
            predictionCol="prediction",
            metricName=metric_name
        )

        metric_values[metric_name] = evaluator.evaluate(
            predictions
        )

    return metric_values, predictions

In [84]:
model_candidates = [
    {
        "model": "Linear regression",
        "reg_param": 0.0,
        "elastic_net": 0.0
    },

    {
        "model": "Ridge",
        "reg_param": 0.0001,
        "elastic_net": 0.0
    },
    {
        "model": "Ridge",
        "reg_param": 0.001,
        "elastic_net": 0.0
    },
    {
        "model": "Ridge",
        "reg_param": 0.01,
        "elastic_net": 0.0
    },
    {
        "model": "Ridge",
        "reg_param": 0.1,
        "elastic_net": 0.0
    },
    {
        "model": "Ridge",
        "reg_param": 1.0,
        "elastic_net": 0.0
    },
    {
        "model": "Ridge",
        "reg_param": 10.0,
        "elastic_net": 0.0
    },

    {
        "model": "Lasso",
        "reg_param": 0.0001,
        "elastic_net": 1.0
    },
    {
        "model": "Lasso",
        "reg_param": 0.001,
        "elastic_net": 1.0
    },
    {
        "model": "Lasso",
        "reg_param": 0.01,
        "elastic_net": 1.0
    },
    {
        "model": "Lasso",
        "reg_param": 0.1,
        "elastic_net": 1.0
    },
    {
        "model": "Lasso",
        "reg_param": 1.0,
        "elastic_net": 1.0
    },
    {
        "model": "Lasso",
        "reg_param": 10.0,
        "elastic_net": 1.0
    }
]

In [139]:
validation_results = []
candidate_models = {}

for candidate in model_candidates:
    model_name = candidate["model"]
    reg_param = candidate["reg_param"]
    elastic_net = candidate["elastic_net"]

    pipeline = create_regression_pipeline(
        label_col=LABEL,
        reg_param=reg_param,
        elastic_net_param=elastic_net
    )

    fitted_pipeline = pipeline.fit(train_data)

    metrics, _ = regression_metrics(
        fitted_pipeline,
        validation_data,
        LABEL
    )

    model_key = (
        model_name,
        reg_param,
        elastic_net
    )

    candidate_models[model_key] = fitted_pipeline

    validation_results.append({
        "model": model_name,
        "reg_param": float(reg_param),
        "elastic_net": float(elastic_net),
        "validation_rmse": float(metrics["rmse"]),
        "validation_mae": float(metrics["mae"]),
        "validation_r2": float(metrics["r2"])
    })

26/07/11 01:33:07 WARN Instrumentation: [81abeb89] regParam is zero, which might cause numerical instability and overfitting.
26/07/11 01:33:07 WARN Instrumentation: [81abeb89] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


In [140]:
validation_results_df = (
    spark.createDataFrame(validation_results)
    .orderBy("validation_rmse")
)

validation_results_df.show(
    100,
    truncate=False
)

+-----------+-----------------+---------+------------------+----------------------+------------------+
|elastic_net|model            |reg_param|validation_mae    |validation_r2         |validation_rmse   |
+-----------+-----------------+---------+------------------+----------------------+------------------+
|1.0        |Lasso            |0.1      |3.558193037916339 |0.4039261213682659    |4.435803737890841 |
|0.0        |Ridge            |10.0     |3.555335982131466 |0.3987704902443876    |4.454945768973974 |
|1.0        |Lasso            |1.0      |3.688120339707216 |0.38993857275091925   |4.487547520641759 |
|0.0        |Ridge            |1.0      |3.536289492024605 |0.3875163092457752    |4.496447652291112 |
|1.0        |Lasso            |0.01     |3.647843890304489 |0.3544123035237078    |4.616362450978999 |
|0.0        |Ridge            |0.1      |3.6528414014617994|0.3494304827913124    |4.6341398138175665|
|0.0        |Ridge            |0.01     |3.6977922678984294|0.32833950777

In [141]:
best_models = {}

for model_family in [
    "Linear regression",
    "Ridge",
    "Lasso"
]:
    family_results = [
        result
        for result in validation_results
        if result["model"] == model_family
    ]

    best_models[model_family] = min(
        family_results,
        key=lambda row: row["validation_rmse"]
    )

best_models

{'Linear regression': {'model': 'Linear regression',
  'reg_param': 0.0,
  'elastic_net': 0.0,
  'validation_rmse': 4.734047795102213,
  'validation_mae': 3.7166519331095014,
  'validation_r2': 0.3210766869092081},
 'Ridge': {'model': 'Ridge',
  'reg_param': 10.0,
  'elastic_net': 0.0,
  'validation_rmse': 4.454945768973974,
  'validation_mae': 3.555335982131466,
  'validation_r2': 0.3987704902443876},
 'Lasso': {'model': 'Lasso',
  'reg_param': 0.1,
  'elastic_net': 1.0,
  'validation_rmse': 4.435803737890841,
  'validation_mae': 3.558193037916339,
  'validation_r2': 0.4039261213682659}}

In [142]:
best_models_df = spark.createDataFrame(
    list(best_models.values())
)

best_models_df.orderBy(
    "validation_rmse"
).show(truncate=False)

+-----------+-----------------+---------+------------------+------------------+-----------------+
|elastic_net|model            |reg_param|validation_mae    |validation_r2     |validation_rmse  |
+-----------+-----------------+---------+------------------+------------------+-----------------+
|1.0        |Lasso            |0.1      |3.558193037916339 |0.4039261213682659|4.435803737890841|
|0.0        |Ridge            |10.0     |3.555335982131466 |0.3987704902443876|4.454945768973974|
|0.0        |Linear regression|0.0      |3.7166519331095014|0.3210766869092081|4.734047795102213|
+-----------+-----------------+---------+------------------+------------------+-----------------+



In [143]:
train_validation_data = train_data.unionByName(
    validation_data
)

final_models = {}
test_results = []
test_prediction_sets = {}

for model_family, specification in best_models.items():
    final_pipeline = create_regression_pipeline(
        label_col=LABEL,
        reg_param=specification["reg_param"],
        elastic_net_param=specification["elastic_net"]
    )

    fitted_final_model = final_pipeline.fit(
        train_validation_data
    )

    metrics, predictions = regression_metrics(
        fitted_final_model,
        test_data,
        LABEL
    )

    final_models[model_family] = fitted_final_model
    test_prediction_sets[model_family] = predictions

    test_results.append({
        "model": model_family,
        "reg_param": float(
            specification["reg_param"]
        ),
        "elastic_net": float(
            specification["elastic_net"]
        ),
        "test_rmse": float(metrics["rmse"]),
        "test_mae": float(metrics["mae"]),
        "test_r2": float(metrics["r2"])
    })

26/07/11 01:33:19 WARN Instrumentation: [b966997d] regParam is zero, which might cause numerical instability and overfitting.
26/07/11 01:33:19 WARN Instrumentation: [b966997d] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


In [144]:
import pandas as pd
import numpy as np

lasso_pipeline_model = final_models["Lasso"]

# Final stage is the fitted LinearRegression model
lasso_model = lasso_pipeline_model.stages[-1]

# Transform one row so Spark creates feature-vector metadata
transformed_sample = lasso_pipeline_model.transform(
    train_validation_data.limit(1)
)

feature_metadata = (
    transformed_sample
    .schema["features"]
    .metadata["ml_attr"]["attrs"]
)

feature_attributes = []

for attribute_type in ["numeric", "binary", "nominal"]:
    feature_attributes.extend(
        feature_metadata.get(attribute_type, [])
    )

feature_attributes = sorted(
    feature_attributes,
    key=lambda attribute: attribute["idx"]
)

feature_names = [
    attribute["name"]
    for attribute in feature_attributes
]

coefficients = lasso_model.coefficients.toArray()

lasso_coefficients = pd.DataFrame({
    "predictor": feature_names,
    "coefficient": coefficients
})

lasso_coefficients["absolute_coefficient"] = (
    lasso_coefficients["coefficient"].abs()
)

In [145]:
SELECTED_TOLERANCE = 1e-8

selected_predictors = (
    lasso_coefficients
    .loc[
        lasso_coefficients["absolute_coefficient"]
        > SELECTED_TOLERANCE
    ]
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

selected_predictors

,predictor,coefficient,absolute_coefficient
0,qualifying_position_percentile_imputed,3.677992,3.677992
1,team_name_ohe_Racing Bulls,0.677258,0.677258
2,team_name_ohe_Mercedes,-0.555903,0.555903
3,team_name_ohe_Red Bull Racing,-0.419789,0.419789
4,team_name_ohe_Williams,0.348159,0.348159
5,team_name_ohe_Haas F1 Team,0.329715,0.329715
6,team_name_ohe_Alfa Romeo,0.288563,0.288563
7,team_name_ohe_AlphaTauri,0.215151,0.215151
8,qualifying_position_imputed,0.208060,0.208060
9,team_name_ohe_McLaren,-0.133078,0.133078


In [92]:
from pyspark.sql import functions as F

base_modeling_df = spark.table(
    "formula1.driver_race_modeling"
)

print("Rows:", base_modeling_df.count())
print("Years:")

base_modeling_df.select("year").distinct().orderBy("year").show()

Rows: 1906
Years:
+----+
|year|
+----+
|2020|
|2021|
|2022|
|2023|
|2025|
|2026|
+----+



In [94]:
results = spark.table("formula1.results")

results_columns = set(results.columns)

if "FullName" in results_columns:
    driver_name_expression = F.col("FullName")

elif {"FirstName", "LastName"}.issubset(results_columns):
    driver_name_expression = F.concat_ws(
        " ",
        F.col("FirstName"),
        F.col("LastName")
    )

elif "BroadcastName" in results_columns:
    driver_name_expression = F.col("BroadcastName")

else:
    raise ValueError(
        "No usable driver-name column was found."
    )

driver_name_lookup = (
    results
    .filter(
        F.upper(
            F.trim(F.col("type_of_race"))
        ) == "R"
    )
    .select(
        F.col("year").cast("int").alias("year"),

        F.col("session_number")
        .cast("int")
        .alias("session_number"),

        F.col("DriverNumber")
        .cast("string")
        .alias("driver_number"),

        F.trim(
            driver_name_expression
        ).alias("driver_name")
    )
    .filter(F.col("driver_name").isNotNull())
    .dropDuplicates(
        [
            "year",
            "session_number",
            "driver_number"
        ]
    )
)

In [122]:
ratings_text = """
Driver	Rating	Year
Lewis Hamilton	94	22
Verstappen	94	22
Charles Leclerc	92	22
George Russell	90	22
Lando Norris	90	22
Fernando Alonso	89	22
Sergio Perez	88	22
Valtteri Bottas	88	22
Carlos Sainz	87	22
Sebastian Vettel	85	22
Pierre Gasly	84	22
Esteban Ocon	83	22
Daniel Ricciardo	83	22
Alex Albon	82	22
Kevin Magnussen	81	22
Lance Stroll	80	22
Yuki Tsuonda	78	22
Mick Schumacher	77	22
Zhou Guanyu	70	22
Nicholas Latifi	70	22
Lewis Hamilton	94	21
Valtteri Bottas	90	21
Max Verstappen	93	21
Sergio Perez	86	21
Charles Leclerc	89	21
Carlos Sainz Jr.	89	21
Sebastian Vettel	89	21
Lance Stroll	82	21
Daniel Ricciardo	89	21
Lando Norris	89	21
Fernando Alonso	89	21
Esteban Ocon	83	21
Pierre Gasly	89	21
Yuki Tsunoda	77	21
Kimi Raikkonen	87	21
Antonio Giovinazzi	81	21
Mick Schumacher	80	21
Nikita Mazepin	67	21
George Russell	81	21
Nicholas Latifi	72	21
Lewis Hamilton	94	20
Valtteri Bottas	90	20
Max Verstappen	90	20
Charles Leclerc	86	20
Sebastian Vettel	89	20
Carlos Sainz Jr.	82	20
Pierre Gasly	80	20
Alexander Albon	79	20
Daniel Ricciardo	87	20
Sergio Perez	85	20
Lando Norris	79	20
Kimi Raikkonen	87	20
Daniil Kvyat	80	20
Lance Stroll	78	20
Kevin Magnussen	78	20
Antonio Giovinazzi	73	20
Romain Grosjean	80	20
George Russell	75	20
Esteban Ocon	80	20
Nicholas Latifi	64	20
Max Verstappen	95	26
Lewis Hamilton	92	26
George Russell	92	26
Charles Leclerc	92	26
Lando Norris	92	26
Oscar Piastri	91	26
Fernando Alonso	89	26
Kimi Antonelli	88	26
Pierre Gasly	85	26
Carlos Sainz	85	26
Nico Hülkenberg	85	26
Sergio Perez	85	26
Isack Hadjar	84	26
Esteban Ocon	84	26
Oliver Bearman	83	26
Alexander Albon	83	26
Valtteri Bottas	82	26
Gabriel Bortoleto	81	26
Liam Lawson	80	26
Lance Stroll	77	26
Franco Colapinto	73	26
Arvid Lindblad	73	26
Max Verstappen	96	24
Fernando Alonso	92	24
Carlos Sainz	89	24
Charles Leclerc	89	24
Lando Norris	89	24
Lewis Hamilton	89	24
George Russell	87	24
Sergio Perez	87	24
Alex Albon	85	24
Oscar Piastri	84	24
Pierre Gasly	84	24
Esteban Ocon	83	24
Daniel Ricciardo	82	24
Nico Hulkenberg	82	24
Valtteri Bottas	81	24
Yuki Tsunoda	81	24
Kevin Magnussen	80	24
Lance Stroll	80	24
Zhou Guanyu	80	24
Logan Sargeant	70	24
Max Verstappen	96	23
Lewis Hamilton	92	23
Fernando Alonso	92	23
Charles Leclerc	90	23
Carlos Sainz Jr.	90	23
Sergio Perez	87	23
George Russell	87	23
Lando Norris	89	23
Valtteri Bottas	87	23
Esteban Ocon	86	23
Pierre Gasly	85	23
Oscar Piastri	85	23
Lance Stroll	84	23
Alexander Albon	83	23
Yuki Tsunoda	83	23
Kevin Magnussen	81	23
Nico Hulkenberg	80	23
Zhou Guanyu	78	23
Logan Sargeant	71	23
Nyck de Vries	71	23
Max Verstappen	95	25
Lando Norris	92	25
Charles Leclerc	92	25
George Russell	92	25
Oscar Piastri	91	25
Lewis Hamilton	91	25
Fernando Alonso	89	25
Nico Hülkenberg	86	25
Alexander Albon	85	25
Pierre Gasly	85	25
Carlos Sainz	85	25
Esteban Ocon	83	25
Isack Hadjar	82	25
Kimi Antonelli	80	25
Oliver Bearman	80	25
Gabriel Bortoleto	79	25
Yuki Tsunoda	77	25
Liam Lawson	77	25
Lance Stroll	77	25
Franco Colapinto	73	25
"""

In [123]:
import pandas as pd
from io import StringIO

ratings_pd = pd.read_csv(
    StringIO(ratings_text.strip()),
    sep="\t"
)

ratings_pd = ratings_pd.rename(
    columns={
        "Driver": "rating_driver_name",
        "Rating": "driver_rating",
        "Year": "year"
    }
)

ratings_pd["year"] = ratings_pd["year"].astype(int)

ratings_pd["year"] = ratings_pd["year"].apply(
    lambda year: year + 2000
    if year < 100
    else year
)

ratings_pd["driver_rating"] = (
    ratings_pd["driver_rating"]
    .astype(float)
)

ratings_pd.head()

,rating_driver_name,driver_rating,year
0,Lewis Hamilton,94.0,2022
1,Verstappen,94.0,2022
2,Charles Leclerc,92.0,2022
3,George Russell,90.0,2022
4,Lando Norris,90.0,2022


In [124]:
import re
import unicodedata

name_aliases = {
    "verstappen": "max verstappen",

    "alex albon": "alexander albon",

    "carlos sainz jr": "carlos sainz",

    "zhou guanyu": "guanyu zhou",

    "yuki tsuonda": "yuki tsunoda",

    "kimi antonelli": "andrea kimi antonelli",

    "nico hulkenberg": "nico hulkenberg",
    "kimi raikkonen": "kimi raikkonen"
}


def normalize_driver_name(name):
    if name is None:
        return None

    normalized = unicodedata.normalize(
        "NFKD",
        str(name)
    )

    normalized = "".join(
        character
        for character in normalized
        if not unicodedata.combining(character)
    )

    normalized = normalized.lower()

    normalized = re.sub(
        r"[^a-z0-9 ]",
        " ",
        normalized
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized
    ).strip()

    return name_aliases.get(
        normalized,
        normalized
    )

In [125]:
ratings_pd["normalized_driver_name"] = (
    ratings_pd["rating_driver_name"]
    .apply(normalize_driver_name)
)

ratings_pd[
    [
        "year",
        "rating_driver_name",
        "normalized_driver_name",
        "driver_rating"
    ]
].head(30)

ratings_spark = spark.createDataFrame(
    ratings_pd[
        [
            "year",
            "rating_driver_name",
            "normalized_driver_name",
            "driver_rating"
        ]
    ]
)

ratings_spark.show(30, truncate=False)

+----+------------------+----------------------+-------------+
|year|rating_driver_name|normalized_driver_name|driver_rating|
+----+------------------+----------------------+-------------+
|2022|Lewis Hamilton    |lewis hamilton        |94.0         |
|2022|Verstappen        |max verstappen        |94.0         |
|2022|Charles Leclerc   |charles leclerc       |92.0         |
|2022|George Russell    |george russell        |90.0         |
|2022|Lando Norris      |lando norris          |90.0         |
|2022|Fernando Alonso   |fernando alonso       |89.0         |
|2022|Sergio Perez      |sergio perez          |88.0         |
|2022|Valtteri Bottas   |valtteri bottas       |88.0         |
|2022|Carlos Sainz      |carlos sainz          |87.0         |
|2022|Sebastian Vettel  |sebastian vettel      |85.0         |
|2022|Pierre Gasly      |pierre gasly          |84.0         |
|2022|Esteban Ocon      |esteban ocon          |83.0         |
|2022|Daniel Ricciardo  |daniel ricciardo      |83.0   

In [126]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

normalize_driver_name_udf = udf(
    normalize_driver_name,
    StringType()
)

driver_name_lookup = (
    driver_name_lookup
    .withColumn(
        "normalized_driver_name",
        normalize_driver_name_udf(
            "driver_name"
        )
    )
)

In [127]:
modeling_with_names = (
    base_modeling_df
    .join(
        driver_name_lookup,
        on=[
            "year",
            "session_number",
            "driver_number"
        ],
        how="left"
    )
)

In [128]:
modeling_with_ratings_check = (
    modeling_with_names
    .join(
        ratings_spark.select(
            "year",
            "normalized_driver_name",
            "driver_rating"
        ),
        on=[
            "year",
            "normalized_driver_name"
        ],
        how="left"
    )
)

In [129]:
unmatched_ratings = (
    modeling_with_ratings_check
    .filter(
        (F.col("year") >= 2020)
        & F.col("driver_rating").isNull()
    )
    .select(
        "year",
        "driver_name",
        "normalized_driver_name"
    )
    .distinct()
    .orderBy(
        "year",
        "driver_name"
    )
)

unmatched_ratings.show(
    100,
    truncate=False
)

+----+-----------------+----------------------+
|year|driver_name      |normalized_driver_name|
+----+-----------------+----------------------+
|2020|Jack Aitken      |jack aitken           |
|2020|Nico Hulkenberg  |nico hulkenberg       |
|2020|Pietro Fittipaldi|pietro fittipaldi     |
|2021|Robert Kubica    |robert kubica         |
|2022|Nico Hulkenberg  |nico hulkenberg       |
|2022|Nyck De Vries    |nyck de vries         |
|2025|Jack Doohan      |jack doohan           |
+----+-----------------+----------------------+



In [130]:
ratings_spark.groupBy(
    "year",
    "normalized_driver_name"
).count().filter(
    F.col("count") > 1
).show(truncate=False)

+----+----------------------+-----+
|year|normalized_driver_name|count|
+----+----------------------+-----+
+----+----------------------+-----+



In [131]:
driver_rating_modeling_df = (
    modeling_with_ratings_check
    .filter(
        F.col("driver_rating").isNotNull()
    )
)

In [132]:
driver_rating_modeling_df = (
    modeling_with_ratings_check
    .filter(
        F.col("driver_rating").isNotNull()
    )
)

In [133]:
driver_rating_modeling_df.groupBy(
    "year"
).agg(
    F.count("*").alias("rows"),
    F.countDistinct("driver_name").alias(
        "drivers"
    ),
    F.min("driver_rating").alias(
        "minimum_rating"
    ),
    F.max("driver_rating").alias(
        "maximum_rating"
    )
).orderBy("year").show()

+----+----+-------+--------------+--------------+
|year|rows|drivers|minimum_rating|maximum_rating|
+----+----+-------+--------------+--------------+
|2020| 334|     20|          64.0|          94.0|
|2021| 436|     20|          67.0|          94.0|
|2022| 436|     20|          70.0|          94.0|
|2023|  60|     20|          71.0|          96.0|
|2025| 472|     21|          73.0|          95.0|
|2026| 151|     22|          73.0|          95.0|
+----+----+-------+--------------+--------------+



In [135]:
spark.sql("""
DROP TABLE IF EXISTS
formula1.driver_race_modeling_with_ratings
""")

DataFrame[]

In [136]:
(
    driver_rating_modeling_df
    .write
    .mode("overwrite")
    .format("parquet")
    .saveAsTable(
        "formula1.driver_race_modeling_with_ratings"
    )
)

In [113]:
requested_numeric_features = [
    "session_number",
    "driver_rating",

    "qualifying_position",
    "qualifying_field_size",
    "qualifying_position_percentile",
    "qualified_top_10",
    "teammate_qualifying_position",
    "qualifying_vs_teammate",

    "valid_qualifying_laps",
    "best_qualifying_lap",
    "median_qualifying_lap",
    "mean_qualifying_lap",
    "qualifying_lap_sd",
    "qualifying_gap_to_pole",

    "maximum_speed_i1",
    "maximum_speed_i2",
    "maximum_finish_line_speed",
    "maximum_speed_trap",

    "qualifying_stints",
    "qualifying_compounds",

    "average_qualifying_position_last_3",
    "average_qualifying_position_last_5",
    "average_gap_to_pole_last_5",

    "race_air_temperature",
    "race_track_temperature",
    "race_humidity",
    "race_pressure",
    "race_wind_speed",
    "race_rain",
    "race_rain_fraction",

    "previous_finish",
    "previous_points",
    "previous_dnf",

    "average_finish_last_3",
    "average_finish_last_5",
    "finish_sd_last_5",

    "average_points_last_5",
    "total_points_last_5",
    "dnf_rate_last_5",
    "average_positions_gained_last_5",

    "season_points_before_race",
    "career_races_before",

    "team_average_finish_last_3",
    "team_average_finish_last_5",
    "team_average_points_last_5",
    "team_dnf_rate_last_5"
]

In [137]:
categorical_features = [
    "team_name"
]

In [138]:
identifier_columns = [
    "year",
    "session_number",
    "driver_number",
    "driver_id",
    "driver_name",
    "normalized_driver_name"
]

In [146]:
# ============================================================
# FORMULA 1: LINEAR REGRESSION, RIDGE, AND LASSO PIPELINE
# ============================================================

from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Imputer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

SOURCE_TABLE = "formula1.driver_race_modeling_with_ratings"
LABEL = "target_position"

# Preferred chronological split.
# Change these if you want to test different seasons.
PREFERRED_VALIDATION_YEAR = 2024
PREFERRED_TEST_YEAR = 2025

# Ridge and lasso penalty values to test.
REGULARIZATION_GRID = [
    0.0001,
    0.001,
    0.01,
    0.1,
    1.0,
    10.0
]

# Coefficients smaller than this are treated as zero.
LASSO_SELECTION_TOLERANCE = 1e-8


# ------------------------------------------------------------
# 2. LOAD THE NEW MODELING DATASET
# ------------------------------------------------------------

model_data = (
    spark.table(SOURCE_TABLE)
    .filter(F.col(LABEL).isNotNull())
)

print("Source table:", SOURCE_TABLE)
print("Total rows:", model_data.count())
print("Total columns:", len(model_data.columns))


# ------------------------------------------------------------
# 3. DEFINE REQUESTED PREDICTORS
# ------------------------------------------------------------

requested_numeric_features = [
    "session_number",

    # Numeric replacement for driver identity
    "driver_rating",

    # Current qualifying results
    "qualifying_position",
    "qualifying_field_size",
    "qualifying_position_percentile",
    "qualified_top_10",
    "teammate_qualifying_position",
    "qualifying_vs_teammate",

    # Current qualifying lap performance
    "valid_qualifying_laps",
    "best_qualifying_lap",
    "median_qualifying_lap",
    "mean_qualifying_lap",
    "qualifying_lap_sd",
    "qualifying_gap_to_pole",

    # Qualifying speed
    "maximum_speed_i1",
    "maximum_speed_i2",
    "maximum_finish_line_speed",
    "maximum_speed_trap",

    # Qualifying session usage
    "qualifying_stints",
    "qualifying_compounds",

    # Historical qualifying form
    "average_qualifying_position_last_3",
    "average_qualifying_position_last_5",
    "average_gap_to_pole_last_5",

    # Actual race weather
    "race_air_temperature",
    "race_track_temperature",
    "race_humidity",
    "race_pressure",
    "race_wind_speed",
    "race_max_wind_speed",
    "race_min_air_temperature",
    "race_max_air_temperature",
    "race_rain",
    "race_rain_fraction",

    # Previous race
    "previous_finish",
    "previous_points",
    "previous_dnf",

    # Historical driver form
    "average_finish_last_3",
    "average_finish_last_5",
    "finish_sd_last_5",
    "average_points_last_5",
    "total_points_last_5",
    "dnf_rate_last_5",
    "average_positions_gained_last_5",

    # Season and career
    "season_points_before_race",
    "career_races_before",

    # Historical constructor form
    "team_average_finish_last_3",
    "team_average_finish_last_5",
    "team_average_points_last_5",
    "team_dnf_rate_last_5"
]

# Driver identity is intentionally excluded.
requested_categorical_features = [
    "team_name"
]

identifier_columns = [
    "year",
    "session_number",
    "driver_number",
    "driver_id",
    "driver_name",
    "normalized_driver_name",
    "team_name"
]


# ------------------------------------------------------------
# 4. KEEP ONLY COLUMNS THAT EXIST
# ------------------------------------------------------------

numeric_features = [
    column
    for column in requested_numeric_features
    if column in model_data.columns
]

categorical_features = [
    column
    for column in requested_categorical_features
    if column in model_data.columns
]

missing_numeric_features = sorted(
    set(requested_numeric_features)
    - set(numeric_features)
)

print("\nNumeric features found:", len(numeric_features))
print("Categorical features found:", categorical_features)

if missing_numeric_features:
    print("\nMissing optional numeric features:")
    for column in missing_numeric_features:
        print(" -", column)


# ------------------------------------------------------------
# 5. CAST NUMERIC FEATURES AND LABEL TO DOUBLE
# ------------------------------------------------------------

for column in numeric_features:
    model_data = model_data.withColumn(
        column,
        F.col(column).cast("double")
    )

model_data = model_data.withColumn(
    LABEL,
    F.col(LABEL).cast("double")
)


# ------------------------------------------------------------
# 6. DETERMINE AVAILABLE YEARS
# ------------------------------------------------------------

available_years = [
    row["year"]
    for row in (
        model_data
        .select(F.col("year").cast("int").alias("year"))
        .filter(F.col("year").isNotNull())
        .distinct()
        .orderBy("year")
        .collect()
    )
]

if len(available_years) < 3:
    raise ValueError(
        "At least three years are required for "
        "training, validation, and testing."
    )

if (
    PREFERRED_VALIDATION_YEAR in available_years
    and PREFERRED_TEST_YEAR in available_years
):
    VALIDATION_YEAR = PREFERRED_VALIDATION_YEAR
    TEST_YEAR = PREFERRED_TEST_YEAR
else:
    VALIDATION_YEAR = available_years[-2]
    TEST_YEAR = available_years[-1]

print("\nAvailable years:", available_years)
print("Validation year:", VALIDATION_YEAR)
print("Test year:", TEST_YEAR)


# ------------------------------------------------------------
# 7. CREATE CHRONOLOGICAL DATA SPLITS
# ------------------------------------------------------------

train_data = (
    model_data
    .filter(F.col("year") < VALIDATION_YEAR)
    .cache()
)

validation_data = (
    model_data
    .filter(F.col("year") == VALIDATION_YEAR)
    .cache()
)

test_data = (
    model_data
    .filter(F.col("year") == TEST_YEAR)
    .cache()
)

train_rows = train_data.count()
validation_rows = validation_data.count()
test_rows = test_data.count()

print("\nTraining rows:", train_rows)
print("Validation rows:", validation_rows)
print("Test rows:", test_rows)

if train_rows == 0:
    raise ValueError("The training dataset is empty.")

if validation_rows == 0:
    raise ValueError("The validation dataset is empty.")

if test_rows == 0:
    raise ValueError("The test dataset is empty.")


# ------------------------------------------------------------
# 8. REMOVE FEATURES THAT ARE COMPLETELY NULL IN TRAINING
# ------------------------------------------------------------

non_null_counts_row = (
    train_data
    .select(
        *[
            F.count(F.col(column)).alias(column)
            for column in numeric_features
        ]
    )
    .first()
)

all_null_features = [
    column
    for column in numeric_features
    if non_null_counts_row[column] == 0
]

numeric_features = [
    column
    for column in numeric_features
    if column not in all_null_features
]

if all_null_features:
    print("\nRemoved all-null training features:")
    for column in all_null_features:
        print(" -", column)

print("\nFinal numeric feature count:", len(numeric_features))


# ------------------------------------------------------------
# 9. CREATE A REUSABLE REGRESSION PIPELINE
# ------------------------------------------------------------

def create_regression_pipeline(
    label_col,
    reg_param,
    elastic_net_param
):
    """
    elasticNetParam:
        0.0 = ridge penalty
        1.0 = lasso penalty

    regParam:
        0.0 = unregularized linear regression
        >0  = regularized regression
    """

    imputed_feature_names = [
        f"{column}_imputed"
        for column in numeric_features
    ]

    imputer = Imputer(
        inputCols=numeric_features,
        outputCols=imputed_feature_names,
        strategy="median"
    )

    indexers = [
        StringIndexer(
            inputCol=column,
            outputCol=f"{column}_index",
            handleInvalid="keep"
        )
        for column in categorical_features
    ]

    pipeline_stages = [
        imputer,
        *indexers
    ]

    encoded_feature_names = []

    if categorical_features:
        encoded_feature_names = [
            f"{column}_ohe"
            for column in categorical_features
        ]

        encoder = OneHotEncoder(
            inputCols=[
                f"{column}_index"
                for column in categorical_features
            ],
            outputCols=encoded_feature_names,
            handleInvalid="keep",
            dropLast=True
        )

        pipeline_stages.append(encoder)

    assembler = VectorAssembler(
        inputCols=(
            imputed_feature_names
            + encoded_feature_names
        ),
        outputCol="features",
        handleInvalid="keep"
    )

    regression = LinearRegression(
        featuresCol="features",
        labelCol=label_col,
        predictionCol="prediction",
        regParam=float(reg_param),
        elasticNetParam=float(elastic_net_param),
        fitIntercept=True,
        standardization=True,
        maxIter=1000,
        tol=1e-6,
        solver="auto"
    )

    pipeline_stages.extend([
        assembler,
        regression
    ])

    return Pipeline(stages=pipeline_stages)


# ------------------------------------------------------------
# 10. METRIC FUNCTION
# ------------------------------------------------------------

def evaluate_regression_model(
    fitted_model,
    data,
    label_col
):
    predictions = fitted_model.transform(data)

    metrics = {}

    for metric_name in ["rmse", "mae", "r2"]:
        evaluator = RegressionEvaluator(
            labelCol=label_col,
            predictionCol="prediction",
            metricName=metric_name
        )

        metrics[metric_name] = float(
            evaluator.evaluate(predictions)
        )

    return metrics, predictions


# ------------------------------------------------------------
# 11. DEFINE MODEL CANDIDATES
# ------------------------------------------------------------

model_candidates = [
    {
        "model_family": "SLR",
        "reg_param": 0.0,
        "elastic_net": 0.0
    }
]

for penalty in REGULARIZATION_GRID:
    model_candidates.append({
        "model_family": "Ridge",
        "reg_param": penalty,
        "elastic_net": 0.0
    })

for penalty in REGULARIZATION_GRID:
    model_candidates.append({
        "model_family": "Lasso",
        "reg_param": penalty,
        "elastic_net": 1.0
    })


# ------------------------------------------------------------
# 12. FIT CANDIDATES ON TRAINING DATA
#     AND EVALUATE ON VALIDATION DATA
# ------------------------------------------------------------

validation_results = []
candidate_models = {}

for candidate in model_candidates:
    model_family = candidate["model_family"]
    reg_param = candidate["reg_param"]
    elastic_net = candidate["elastic_net"]

    print(
        f"Fitting {model_family}: "
        f"regParam={reg_param}, "
        f"elasticNetParam={elastic_net}"
    )

    pipeline = create_regression_pipeline(
        label_col=LABEL,
        reg_param=reg_param,
        elastic_net_param=elastic_net
    )

    fitted_model = pipeline.fit(train_data)

    metrics, _ = evaluate_regression_model(
        fitted_model,
        validation_data,
        LABEL
    )

    model_key = (
        model_family,
        reg_param,
        elastic_net
    )

    candidate_models[model_key] = fitted_model

    validation_results.append({
        "model": model_family,
        "reg_param": float(reg_param),
        "elastic_net": float(elastic_net),
        "validation_rmse": metrics["rmse"],
        "validation_mae": metrics["mae"],
        "validation_r2": metrics["r2"]
    })


# ------------------------------------------------------------
# 13. DISPLAY ALL VALIDATION RESULTS
# ------------------------------------------------------------

validation_results_pd = (
    pd.DataFrame(validation_results)
    .sort_values(
        ["validation_rmse", "validation_mae"]
    )
    .reset_index(drop=True)
)

print("\nALL VALIDATION RESULTS")
display(validation_results_pd)


# ------------------------------------------------------------
# 14. SELECT THE BEST MODEL IN EACH FAMILY
# ------------------------------------------------------------

best_model_specs = {}

for model_family in ["SLR", "Ridge", "Lasso"]:
    family_results = validation_results_pd[
        validation_results_pd["model"] == model_family
    ]

    best_row = (
        family_results
        .sort_values("validation_rmse")
        .iloc[0]
        .to_dict()
    )

    best_model_specs[model_family] = best_row

best_specs_pd = pd.DataFrame(
    list(best_model_specs.values())
).sort_values("validation_rmse")

print("\nBEST VALIDATION MODEL FROM EACH FAMILY")
display(best_specs_pd)


# ------------------------------------------------------------
# 15. REFIT EACH SELECTED MODEL USING
#     TRAINING + VALIDATION DATA
# ------------------------------------------------------------

train_validation_data = (
    train_data
    .unionByName(validation_data)
    .cache()
)

print(
    "\nTraining + validation rows:",
    train_validation_data.count()
)

final_models = {}
test_prediction_sets = {}
test_results = []

for model_family, specification in best_model_specs.items():

    final_pipeline = create_regression_pipeline(
        label_col=LABEL,
        reg_param=specification["reg_param"],
        elastic_net_param=specification["elastic_net"]
    )

    fitted_final_model = final_pipeline.fit(
        train_validation_data
    )

    metrics, predictions = evaluate_regression_model(
        fitted_final_model,
        test_data,
        LABEL
    )

    final_models[model_family] = fitted_final_model
    test_prediction_sets[model_family] = predictions

    test_results.append({
        "model": model_family,
        "reg_param": float(
            specification["reg_param"]
        ),
        "elastic_net": float(
            specification["elastic_net"]
        ),
        "test_rmse": metrics["rmse"],
        "test_mae": metrics["mae"],
        "test_r2": metrics["r2"]
    })


# ------------------------------------------------------------
# 16. DISPLAY FINAL TEST RESULTS
# ------------------------------------------------------------

test_results_pd = (
    pd.DataFrame(test_results)
    .sort_values(
        ["test_rmse", "test_mae"]
    )
    .reset_index(drop=True)
)

print("\nFINAL TEST RESULTS")
display(test_results_pd)


# ------------------------------------------------------------
# 17. CREATE A COMBINED TEST PREDICTION TABLE
# ------------------------------------------------------------

available_identifier_columns = [
    column
    for column in identifier_columns
    if column in model_data.columns
]

slr_predictions = (
    test_prediction_sets["SLR"]
    .select(
        *available_identifier_columns,
        LABEL,
        F.col("prediction").alias("slr_prediction")
    )
)

ridge_predictions = (
    test_prediction_sets["Ridge"]
    .select(
        *[
            column
            for column in [
                "year",
                "session_number",
                "driver_number"
            ]
            if column in model_data.columns
        ],
        F.col("prediction").alias("ridge_prediction")
    )
)

lasso_predictions = (
    test_prediction_sets["Lasso"]
    .select(
        *[
            column
            for column in [
                "year",
                "session_number",
                "driver_number"
            ]
            if column in model_data.columns
        ],
        F.col("prediction").alias("lasso_prediction")
    )
)

prediction_join_columns = [
    column
    for column in [
        "year",
        "session_number",
        "driver_number"
    ]
    if column in model_data.columns
]

all_test_predictions = (
    slr_predictions
    .join(
        ridge_predictions,
        on=prediction_join_columns,
        how="inner"
    )
    .join(
        lasso_predictions,
        on=prediction_join_columns,
        how="inner"
    )
    .withColumn(
        "slr_absolute_error",
        F.abs(
            F.col(LABEL)
            - F.col("slr_prediction")
        )
    )
    .withColumn(
        "ridge_absolute_error",
        F.abs(
            F.col(LABEL)
            - F.col("ridge_prediction")
        )
    )
    .withColumn(
        "lasso_absolute_error",
        F.abs(
            F.col(LABEL)
            - F.col("lasso_prediction")
        )
    )
)

print("\nTEST PREDICTIONS")

all_test_predictions.orderBy(
    "year",
    "session_number",
    LABEL
).show(
    100,
    truncate=False
)


# ------------------------------------------------------------
# 18. EXTRACT LASSO-SELECTED PREDICTORS
# ------------------------------------------------------------

lasso_pipeline_model = final_models["Lasso"]
lasso_regression_model = lasso_pipeline_model.stages[-1]

lasso_coefficients_array = (
    lasso_regression_model
    .coefficients
    .toArray()
)

# Transform one row so the features column contains metadata.
lasso_transformed_sample = (
    lasso_pipeline_model
    .transform(train_validation_data.limit(1))
)

features_metadata = (
    lasso_transformed_sample
    .schema["features"]
    .metadata
)

feature_names = []

try:
    attribute_groups = (
        features_metadata["ml_attr"]["attrs"]
    )

    feature_attributes = []

    for attribute_type in [
        "numeric",
        "binary",
        "nominal"
    ]:
        feature_attributes.extend(
            attribute_groups.get(
                attribute_type,
                []
            )
        )

    feature_attributes = sorted(
        feature_attributes,
        key=lambda attribute: attribute["idx"]
    )

    feature_names = [
        attribute["name"]
        for attribute in feature_attributes
    ]

except Exception:
    feature_names = [
        f"feature_{index}"
        for index in range(
            len(lasso_coefficients_array)
        )
    ]

# Metadata occasionally omits a feature.
if len(feature_names) != len(lasso_coefficients_array):
    feature_names = [
        f"feature_{index}"
        for index in range(
            len(lasso_coefficients_array)
        )
    ]

lasso_coefficients_pd = pd.DataFrame({
    "predictor": feature_names,
    "coefficient": lasso_coefficients_array
})

lasso_coefficients_pd["absolute_coefficient"] = (
    lasso_coefficients_pd["coefficient"].abs()
)

lasso_coefficients_pd["selected"] = (
    lasso_coefficients_pd["absolute_coefficient"]
    > LASSO_SELECTION_TOLERANCE
)

lasso_coefficients_pd["clean_predictor"] = (
    lasso_coefficients_pd["predictor"]
    .str.replace(
        "_imputed$",
        "",
        regex=True
    )
)

selected_lasso_predictors = (
    lasso_coefficients_pd[
        lasso_coefficients_pd["selected"]
    ]
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

removed_lasso_predictors = (
    lasso_coefficients_pd[
        ~lasso_coefficients_pd["selected"]
    ]
    .sort_values("predictor")
    .reset_index(drop=True)
)

print("\nLASSO FEATURE-SELECTION SUMMARY")
print(
    "Total encoded predictors:",
    len(lasso_coefficients_pd)
)
print(
    "Selected predictors:",
    len(selected_lasso_predictors)
)
print(
    "Removed predictors:",
    len(removed_lasso_predictors)
)

print("\nLASSO-SELECTED PREDICTORS")
display(
    selected_lasso_predictors[
        [
            "clean_predictor",
            "predictor",
            "coefficient",
            "absolute_coefficient"
        ]
    ]
)

print("\nPREDICTORS REMOVED BY LASSO")
display(
    removed_lasso_predictors[
        [
            "clean_predictor",
            "predictor",
            "coefficient"
        ]
    ]
)


# ------------------------------------------------------------
# 19. SHOW ONLY SELECTED NUMERIC PREDICTORS
# ------------------------------------------------------------

selected_numeric_lasso_predictors = (
    selected_lasso_predictors[
        ~selected_lasso_predictors[
            "predictor"
        ].str.startswith(
            tuple(
                f"{column}_ohe"
                for column in categorical_features
            )
        )
    ]
    .reset_index(drop=True)
)

print("\nSELECTED NUMERIC LASSO PREDICTORS")
display(
    selected_numeric_lasso_predictors[
        [
            "clean_predictor",
            "coefficient",
            "absolute_coefficient"
        ]
    ]
)


# ------------------------------------------------------------
# 20. CLEANUP CACHED DATA
# ------------------------------------------------------------

train_data.unpersist()
validation_data.unpersist()
test_data.unpersist()
train_validation_data.unpersist()

Source table: formula1.driver_race_modeling_with_ratings
Total rows: 1889
Total columns: 57

Numeric features found: 49
Categorical features found: ['team_name']

Available years: [2020, 2021, 2022, 2023, 2025, 2026]
Validation year: 2025
Test year: 2026

Training rows: 1266
Validation rows: 472
Test rows: 151

Final numeric feature count: 49
Fitting SLR: regParam=0.0, elasticNetParam=0.0


26/07/11 01:35:36 WARN Instrumentation: [23adbff5] regParam is zero, which might cause numerical instability and overfitting.
26/07/11 01:35:36 WARN Instrumentation: [23adbff5] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


Fitting Ridge: regParam=0.0001, elasticNetParam=0.0
Fitting Ridge: regParam=0.001, elasticNetParam=0.0
Fitting Ridge: regParam=0.01, elasticNetParam=0.0
Fitting Ridge: regParam=0.1, elasticNetParam=0.0
Fitting Ridge: regParam=1.0, elasticNetParam=0.0
Fitting Ridge: regParam=10.0, elasticNetParam=0.0
Fitting Lasso: regParam=0.0001, elasticNetParam=1.0
Fitting Lasso: regParam=0.001, elasticNetParam=1.0
Fitting Lasso: regParam=0.01, elasticNetParam=1.0
Fitting Lasso: regParam=0.1, elasticNetParam=1.0
Fitting Lasso: regParam=1.0, elasticNetParam=1.0
Fitting Lasso: regParam=10.0, elasticNetParam=1.0

ALL VALIDATION RESULTS


,model,reg_param,elastic_net,validation_rmse,validation_mae,validation_r2
0,Lasso,0.1000,1.0,4.330627,3.449685,0.429997
1,Ridge,1.0000,0.0,4.426922,3.512299,0.404366
2,Ridge,10.0000,0.0,4.437643,3.544759,0.401478
3,Lasso,1.0000,1.0,4.462235,3.666109,0.394826
4,Lasso,0.0100,1.0,4.491721,3.590863,0.386802
5,Ridge,0.1000,0.0,4.522806,3.596197,0.378285
6,Ridge,0.0100,0.0,4.587448,3.631775,0.360386
7,Lasso,0.0010,1.0,4.601048,3.641025,0.356588
8,Lasso,0.0001,1.0,4.601598,3.637955,0.356434
9,Ridge,0.0010,0.0,4.606297,3.642265,0.355119



BEST VALIDATION MODEL FROM EACH FAMILY


...


PREDICTORS REMOVED BY LASSO


,clean_predictor,predictor,coefficient
0,average_finish_last_3,average_finish_last_3_imputed,0.0
1,average_finish_last_5,average_finish_last_5_imputed,0.0
2,average_gap_to_pole_last_5,average_gap_to_pole_last_5_imputed,0.0
3,average_points_last_5,average_points_last_5_imputed,0.0
4,average_qualifying_position_last_3,average_qualifying_position_last_3_imputed,0.0
5,average_qualifying_position_last_5,average_qualifying_position_last_5_imputed,0.0
6,best_qualifying_lap,best_qualifying_lap_imputed,0.0
7,career_races_before,career_races_before_imputed,0.0
8,dnf_rate_last_5,dnf_rate_last_5_imputed,0.0
9,finish_sd_last_5,finish_sd_last_5_imputed,0.0



SELECTED NUMERIC LASSO PREDICTORS


,clean_predictor,coefficient,absolute_coefficient
0,qualifying_position,0.370588,0.370588
1,driver_rating,-0.129559,0.129559
2,team_average_finish_last_5,0.082986,0.082986
3,average_positions_gained_last_5,0.081203,0.081203
4,team_average_finish_last_3,0.043087,0.043087
5,previous_finish,0.029521,0.029521
6,teammate_qualifying_position,0.025616,0.025616
7,qualifying_gap_to_pole,-0.012439,0.012439
8,valid_qualifying_laps,0.006842,0.006842
9,season_points_before_race,-0.002856,0.002856


DataFrame[year: int, normalized_driver_name: string, session_number: double, driver_number: string, driver_id: string, team_name: string, qualifying_position: double, qualifying_field_size: double, qualifying_position_percentile: double, qualified_top_10: double, teammate_qualifying_position: double, qualifying_vs_teammate: double, valid_qualifying_laps: double, best_qualifying_lap: double, median_qualifying_lap: double, mean_qualifying_lap: double, qualifying_lap_sd: double, qualifying_gap_to_pole: double, maximum_speed_i1: double, maximum_speed_i2: double, maximum_finish_line_speed: double, maximum_speed_trap: double, qualifying_stints: double, qualifying_compounds: double, average_qualifying_position_last_3: double, average_qualifying_position_last_5: double, average_gap_to_pole_last_5: double, race_air_temperature: double, race_track_temperature: double, race_humidity: double, race_pressure: double, race_wind_speed: double, race_max_wind_speed: double, race_min_air_temperature: dou